<a href="https://colab.research.google.com/github/AhmedGDeeb/DataTalksClub/blob/main/machine-learning-zoomcamp/cohorts/2026/homework/06-trees/homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## setup

In [84]:
!pip install xgboost tqdm scikit-learn pandas

In [85]:
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer

import xgboost as xgb
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

## Utils

## Homework 6

> [!NOTE]
> This homework uses the pinned 2026 car fuel-efficiency release in the course
> repository. The plan and report are available in `cohorts/2026/data/`.


### Dataset

In this homework, we continue using the fuel efficiency dataset.
Download it from <a href='https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'>here</a>.

You can do it with wget:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
```

The goal of this homework is to create a regression model for predicting the car fuel efficiency (column `'fuel_efficiency_mpg'`).


In [86]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv')

### Preparing the dataset

Preparation:

* Fill missing values with zeros.
* Do the split with these exact calls:

```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)
```

* Use `DictVectorizer(sparse=True)` to turn the dataframes into matrices.


In [87]:
target_column = 'fuel_efficiency_mpg'
# fill missing values with zeros
df = df.fillna(0)
# split as demanded
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)
# X, y
X_train = df_train.drop(columns=[target_column])
X_train_dicts = X_train.to_dict(orient='records')
y_train = df_train[target_column]

X_test = df_test.drop(columns=[target_column])
X_test_dicts = X_test.to_dict(orient='records')
y_test = df_test[target_column]

X_val = df_val.drop(columns=[target_column])
X_val_dicts = X_val.to_dict(orient='records')
y_val = df_val[target_column]

# turn df into metrices with DicVectorizer(sparse=True)
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(X_train_dicts)
X_test = dv.transform(X_test_dicts)
X_val = dv.transform(X_val_dicts)
feature_names = dv.get_feature_names_out()

## Question 1



Let's train a decision tree regressor to predict the `fuel_efficiency_mpg` variable.

* Train a model with `max_depth=1`.


Which original feature is used for splitting the data? If you use
`DictVectorizer`, map a one-hot feature such as `origin=USA` back to its
original column name.


* `'vehicle_weight'`
* `'model_year'`
* `'origin'`
* `'fuel_type'`

In [88]:
dt_reg = DecisionTreeRegressor(max_depth=1)
dt_reg.fit(X_train, y_train)
splitting_feature = None
if dt_reg.feature_importances_.tolist().count(1.0) == 1:
    splitting_feature = feature_names[dt_reg.feature_importances_.tolist().index(1.0)]

## Answer

In [89]:
splitting_feature

'model_year'

## Question 2

Train a random forest regressor with these parameters:

* `n_estimators=10`
* `random_state=1`
* `n_jobs=-1` (optional - to make training faster)


What's the RMSE of this model on the validation data?

* 0.1837
* 1.837
* 18.37
* 183.7

In [90]:
rf_reg = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf_reg.fit(X_train, y_train)
y_pred = rf_reg.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)

## Answer

In [91]:
round(rmse, 3)

1.837

## Question 3

Now let's experiment with the `n_estimators` parameter

* Try these values of this parameter: `[10, 50, 100, 150]`.
* Set `random_state` to `1`.
* Evaluate the model on the validation dataset.


Which value gives the lowest validation RMSE? Consider 3 decimal places for
calculating the answer.

- 10
- 50
- 100
- 150

In [92]:
n_estimators = [10, 50, 100, 150]
results = []
for n in tqdm(n_estimators, desc="Tuning n_estimators"):
    rf_reg = RandomForestRegressor(n_estimators=n, random_state=1)
    rf_reg.fit(X_train, y_train)
    y_pred = rf_reg.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    results.append({
        'n_estimators': n,
        'rmse': rmse
        })

results

Tuning n_estimators:   0%|          | 0/4 [00:00<?, ?it/s]

[{'n_estimators': 10, 'rmse': 1.837054054730018},
 {'n_estimators': 50, 'rmse': 1.7715666428333987},
 {'n_estimators': 100, 'rmse': 1.7681362324492986},
 {'n_estimators': 150, 'rmse': 1.7688040324216567}]

## Answer

In [93]:
best = min(results, key=lambda r: r['rmse'])
best['n_estimators']

100

## Question 4

Let's select the best `max_depth`:

* Try different values of `max_depth`: `[10, 15, 20, 25]`
* For each of these values,
  * try different values of `n_estimators` from `[10, 50, 100, 150]`
  * calculate the mean RMSE
* Fix the random seed: `random_state=1`


What's the best `max_depth`, using the mean RMSE?

* 10
* 15
* 20
* 25

In [94]:
max_depth = [10, 15, 20, 25]
n_estimators = [10, 50, 100, 150]
results = []
for m in tqdm(max_depth, desc="max_depth"):
    rmse_per_n = []
    for n in n_estimators:
        rf_reg = RandomForestRegressor(
            n_estimators=n,
            max_depth=m,
            random_state=1,
        )
        rf_reg.fit(X_train, y_train)
        y_pred = rf_reg.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        rmse_per_n.append(rmse)

    mean_rmse = sum(rmse_per_n) / len(rmse_per_n)
    results.append({'max_depth': m, 'mean_rmse': mean_rmse})
    tqdm.write(f"max_depth={m} -> mean RMSE={mean_rmse:.4f}")

results

max_depth:   0%|          | 0/4 [00:00<?, ?it/s]

max_depth=10 -> mean RMSE=1.7554
max_depth=15 -> mean RMSE=1.7840
max_depth=20 -> mean RMSE=1.7882
max_depth=25 -> mean RMSE=1.7863


[{'max_depth': 10, 'mean_rmse': 1.7554047863550186},
 {'max_depth': 15, 'mean_rmse': 1.7839526938716292},
 {'max_depth': 20, 'mean_rmse': 1.788242068569452},
 {'max_depth': 25, 'mean_rmse': 1.786317363879773}]

## Answer

In [95]:
best = min(results, key=lambda r: r['mean_rmse'])
best['max_depth']

10

## Question 5

We can extract feature importance information from tree-based models.

At each step of the decision tree learning algorithm, it finds the best split.
When doing it, we can calculate "gain" - the reduction in impurity before and after the split.
This gain is quite useful in understanding what are the important features for tree-based models.

In Scikit-Learn, tree-based models contain this information in the
[`feature_importances_`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html#sklearn.ensemble.RandomForestRegressor.feature_importances_)
field.

For this homework question, we'll find the most important feature:

* Train the model with these parameters:
  * `n_estimators=10`,
  * `max_depth=20`,
  * `random_state=1`,
  * `n_jobs=-1` (optional)
* Get the feature importance information from this model


What's the most important feature (among these 4)?

* `vehicle_weight`
*	`horsepower`
* `acceleration`
* `engine_displacement`

In [96]:
rf_reg = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf_reg.fit(X_train, y_train)
imp = dict(zip(feature_names, rf_reg.feature_importances_))
imp

{'acceleration': np.float64(0.05520111946946774),
 'drivetrain=All-wheel drive': np.float64(0.007681663098158672),
 'drivetrain=Front-wheel drive': np.float64(0.02014531832543737),
 'drivetrain=Rear-wheel drive': np.float64(0.0057808697447890986),
 'engine_displacement': np.float64(0.06100490942209099),
 'fuel_type=Diesel': np.float64(0.017038345612737704),
 'fuel_type=Gasoline': np.float64(0.16071245499347314),
 'fuel_type=Hybrid': np.float64(0.015799723579542093),
 'horsepower': np.float64(0.07801732200517682),
 'model_year': np.float64(0.29600139070972864),
 'num_cylinders': np.float64(0.007283541644733502),
 'num_doors': np.float64(0.019414978403743424),
 'origin=Asia': np.float64(0.006390907907055007),
 'origin=Europe': np.float64(0.003504506701702119),
 'origin=USA': np.float64(0.04214437633660763),
 'vehicle_weight': np.float64(0.20387857204555604)}

## Answer

In [97]:
candidates = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
top_feature = max(candidates, key=imp.get)
top_feature

'vehicle_weight'

## Question 6

Now let's train an XGBoost model! For this question, we'll tune the `eta` parameter:

* Install XGBoost
* Create DMatrix for train and validation
* Create a watchlist
* Train a model with these parameters for 100 rounds:

```
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
```

Now change `eta` from `0.3` to `0.1`.

Which eta leads to the best RMSE score on the validation dataset?

* 0.3
* 0.1
* Both give equal value

In [98]:
results = []
# DMatrix for train and validation
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=list(feature_names))
dval   = xgb.DMatrix(X_val,   label=y_val, feature_names=list(feature_names))
# watchlist
watchlist = [(dtrain, 'train'), (dval, 'val')]
# train with different etas
etas = [0.3, 0.1]
for eta in tqdm(etas, desc="Tuning eta"):
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }

    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=100,
        evals=watchlist,
        verbose_eval=False,
    )

    y_pred = model.predict(dval)

    rmse = root_mean_squared_error(y_val, y_pred)

    results.append({'eta': eta, 'rmse': rmse})

    tqdm.write(f"eta={eta} -> val RMSE={rmse:.4f}")

results

Tuning eta:   0%|          | 0/2 [00:00<?, ?it/s]

eta=0.3 -> val RMSE=1.8265
eta=0.1 -> val RMSE=1.7248


[{'eta': 0.3, 'rmse': 1.8264579799904583},
 {'eta': 0.1, 'rmse': 1.724810059827976}]

## Answer

In [99]:
best = min(results, key=lambda r: r['rmse'])
best['eta']

0.1

## Submit the results

* Submit your results here: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw06
* The numerical options are calculated from the pinned 2026 release. Use the value that matches your calculation; do not choose a merely close value.
